In [1]:
from pathlib import Path

import numpy as np
import torch
from rtnls_enface import FundusLoader
from rtnls_enface.utils.plotting import plot_gridfns
from rtnls_models import DiscDetector, KeypointsEnsemble

## Segmentation of preprocessed images

Here we segment images preprocessed using 0_preprocess.ipynb

In [2]:
ds_path = Path("../../samples/fundus")

# input folder. point to a folder with png and/or dicom images
npy_path = ds_path / "preprocessed_npy"

# these are the output folders:
av_path = ds_path / "av"
overlays_path = ds_path / "overlays"
discs_path = ds_path / "discs"

device = torch.device("cuda:0")

### Artery-vein segmentation

In [3]:
# set the model to use here. This one is working okay.
av_model = ArteryVeinSegmentor(device=device, model_name="nonmasked_mild-augment_16")
# run the model, pass the path for the outputs
av_model.predict(npy_path, dest_path=av_path, num_workers=2, preprocess=False)

### Disc segmentation

In [4]:
disc_model = DiscDetector(device=device)
disc_model.predict(npy_path, dest_path=discs_path, preprocess=False)

/home/jose/miniconda3/lib/python3.10/site-packages/pytorch_lightning/utilities/migration/migration.py:195: PossibleUserWarning: You have multiple `ModelCheckpoint` callback states in this checkpoint, but we found state keys that would end up colliding with each other after an upgrade, which means we can't differentiate which of your checkpoint callbacks needs which states. At least one of your `ModelCheckpoint` callbacks will not be able to reload the state.
  rank_zero_warn(
Lightning automatically upgraded your loaded checkpoint from v1.3.8 to v1.9.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file ../../rtnls_models/disc/models/latest.ckpt`


[PosixPath('../../samples/fundus/preprocessed_npy/210281.npy'), PosixPath('../../samples/fundus/preprocessed_npy/245938.npy'), PosixPath('../../samples/fundus/preprocessed_npy/360268.npy'), PosixPath('../../samples/fundus/preprocessed_npy/375802.npy'), PosixPath('../../samples/fundus/preprocessed_npy/560772.npy'), PosixPath('../../samples/fundus/preprocessed_npy/604547.npy'), PosixPath('../../samples/fundus/preprocessed_npy/704166.npy'), PosixPath('../../samples/fundus/preprocessed_npy/706168.npy'), PosixPath('../../samples/fundus/preprocessed_npy/803630.npy'), PosixPath('../../samples/fundus/preprocessed_npy/805765.npy')]
FundusDataset created. len(images_or_paths) = 10, len(masks_or_paths)=0, preprocess=False
Filters: [32, 64, 96, 128, 192, 256, 384, 512],
Kernels: [[3, 3], [3, 3], [3, 3], [3, 3], [3, 3], [3, 3], [3, 3], [3, 3]]
Strides: [[1, 1], [2, 2], [2, 2], [2, 2], [2, 2], [2, 2], [2, 2], [2, 2]]


10it [00:01,  7.97it/s]


### Fovea detection

In [5]:
fovea_model = KeypointsEnsemble(device=device, model_name="r101_b16_001")
df = fovea_model.predict(npy_path, preprocess=False)

FundusDataset created. len(images_or_paths) = 10, len(masks_or_paths)=0, preprocess=False


1it [00:00,  1.42it/s]


In [ ]:
df.head(10)

In [ ]:
# the fovea model returns a dataframe with the fovea locations predictions by 5 models (x1,y1),(x2,y2),(x3,y3),(x4,y4),(x5,y5)
# here we average the predictions of the 5 models before storing the dataframe
df["mean_x"] = np.mean([df[f"x{i}"] for i in range(0, 5)], axis=0)
df["mean_y"] = np.mean([df[f"y{i}"] for i in range(0, 5)], axis=0)
df.to_csv(ds_path / "fovea.csv")

### Plotting the retinas

This will only work if you ran all the models and stored the outputs using the same folder/file names as above

In [ ]:
loader = FundusLoader.from_folder(ds_path, fundus_subfolder="preprocessed_rgb")

In [ ]:
plot_gridfns([ret.plot for ret in loader])